# Diabetes Readmission Prediction

Compare linear SVM, logistic regression, random forest, and random forest with feature selection on hospital encounter records. The binary outcome combines readmissions before and after 30 days.

Run cells in order from a fresh kernel. Local CSV files are included under `data/`. Default settings preserve the recorded experiment design; `DIABETES_SMOKE_TEST=1` enables a small infrastructure check, not a comparable experiment.

See [README](README.md) for recorded results and [evaluation notes](docs/evaluation.md) for encounter-level splits, fold-level preprocessing limits, and uncertainty interpretation.

## 1. Explore hospital encounter records

### Dataset structure and descriptive statistics

In [ ]:
# Import the libraries used in this notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Improve table display in the notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

# Set a sensible default figure size
plt.rcParams['figure.figsize'] = (10, 6)

import os
SMOKE_TEST = os.environ.get("DIABETES_SMOKE_TEST", "0") == "1"
RUN_KIND = "smoke_check" if SMOKE_TEST else "full_experiment"
print("Execution mode:", RUN_KIND)


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "diabetic_data.csv").is_file():
    raise FileNotFoundError("Open this notebook from the repository root containing data/.")
data_file = PROJECT_ROOT / "data" / "diabetic_data.csv"
mapping_file = PROJECT_ROOT / "data" / "IDs_mapping.csv"
print("Dataset:", data_file.name)


In [ ]:
# Load strings consistently across pandas versions; retain the original NA parser.
df = pd.read_csv(data_file)
string_columns = df.select_dtypes(include=["object", "string"]).columns
df[string_columns] = df[string_columns].astype(object)
df = df.where(pd.notna(df), np.nan)
print("Loaded encounters:", len(df))
df.head()


In [ ]:
# Check the size of the dataset
# Show the number of rows and columns in the dataset
print('Dataset shape:', df.shape)
print('Number of samples:', df.shape[0])
print('Number of columns (including target and identifiers):', df.shape[1])

In [ ]:
# Show all column names
# Display all column names
print('Column names:\n')
for column in df.columns:
    print(column)

In [ ]:
# Separate the target column from the predictor count.
print("Raw columns:", df.shape[1])
print("Predictor columns before identifier removal:", df.shape[1] - 1)
print("Encounter-to-predictor ratio:", round(len(df) / (df.shape[1] - 1), 2))


In [ ]:
# Inspect the data types of the columns

print('Data types summary:\n')
print(df.dtypes.value_counts())

# Separate numerical and categorical columns
numerical_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_columns = df.select_dtypes(include=['object']).columns.tolist()

print('\nNumber of numerical columns:', len(numerical_columns))
print('Number of categorical columns:', len(categorical_columns))

print('\nNumerical columns:\n')
print(numerical_columns)

print('\nCategorical columns:\n')
print(categorical_columns)

In [ ]:
# Inspect the original readmission target

print(df['readmitted'].value_counts(dropna=False))

print('\nProportions:\n')
print(df['readmitted'].value_counts(normalize=True, dropna=False))

In [ ]:
# Plot the distribution of the original readmission classes

readmitted_counts = df['readmitted'].value_counts()

plt.figure(figsize=(8, 5))
plt.bar(readmitted_counts.index, readmitted_counts.values)
plt.title('Distribution of original readmission classes')
plt.xlabel('Readmission class')
plt.ylabel('Number of encounters')
plt.show()

In [ ]:
# Check standard missing values

missing_counts_standard = df.isna().sum().sort_values(ascending=False)
missing_counts_standard.head(20)

In [ ]:
# Check missing values recorded as question marks

question_mark_counts = (df == '?').sum().sort_values(ascending=False)
question_mark_counts.head(20)

In [ ]:
# Plot the features with the highest number of question-mark entries

top_missing = question_mark_counts[question_mark_counts > 0].head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_missing.index[::-1], top_missing.values[::-1])
plt.title('Top 10 features with missing values coded as question marks')
plt.xlabel('Number of missing entries')
plt.ylabel('Feature')
plt.show()

In [ ]:
# Display summary statistics for numerical features

df[numerical_columns].describe().T

In [ ]:
# Plot the distribution of age groups

age_counts = df['age'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(age_counts.index, age_counts.values)
plt.title('Distribution of age groups')
plt.xlabel('Age group')
plt.ylabel('Number of encounters')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Plot the distribution of the number of inpatient visits

plt.figure(figsize=(8, 5))
plt.hist(df['number_inpatient'], bins=20)
plt.title('Distribution of number of inpatient visits')
plt.xlabel('Number of inpatient visits')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Check the number of unique values in each categorical feature

categorical_unique_counts = df[categorical_columns].nunique().sort_values(ascending=False)
categorical_unique_counts.head(20)

In [ ]:
# Plot the distribution of the binary readmission target

readmitted_clean = df['readmitted'].astype(str).str.strip().str.upper()

binary_readmitted = readmitted_clean.map({
    'NO': 0,
    '<30': 1,
    '>30': 1
})

binary_counts = binary_readmitted.value_counts().reindex([0, 1], fill_value=0)

plt.figure(figsize=(6, 4))
plt.bar(['No readmission', 'Readmission'], binary_counts.values)
plt.title('Distribution of binary readmission target')
plt.ylabel('Number of encounters')
plt.show()

print(binary_counts)
print(binary_counts / binary_counts.sum())

In [ ]:
# Load the mapping file for reference

mapping_df = pd.read_csv(mapping_file)

# Display the first few rows
mapping_df.head()

In [ ]:
# Correlations are descriptive; numeric identifier codes are not clinical quantities.
corr_matrix = df[numerical_columns].corr()
fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
ax.set_yticks(range(len(corr_matrix.index)), corr_matrix.index)
ax.set_title("Correlation matrix of numerically stored columns")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()


### Missingness, cardinality, and class distribution

In [ ]:
# Check all possible "weird" values in categorical columns

for col in categorical_columns:
    unique_vals = df[col].astype(str).str.strip().unique()

    # Print columns that might contain unusual values
    suspicious = [v for v in unique_vals if str(v).lower() in ['?', 'na', 'nan', 'none', 'null', '']]

    if len(suspicious) > 0:
        print(f"{col}: {suspicious}")

In [ ]:
# Plot missing values (including '?')

import matplotlib.pyplot as plt

# Combine NaN and '?' missing
missing_nan = df.isna().sum()
missing_q = (df == '?').sum()

total_missing = missing_nan + missing_q

# Select top 10 features with most missing values
top_missing = total_missing.sort_values(ascending=False).head(10)

# Plot
plt.figure(figsize=(10, 6))
plt.barh(top_missing.index[::-1], top_missing.values[::-1])
plt.title('Top 10 features with missing values')
plt.xlabel('Number of missing entries')
plt.ylabel('Feature')
plt.show()

In [ ]:
# Plot class distribution (better visual)

readmitted_counts = df['readmitted'].value_counts()

plt.figure(figsize=(6, 4))
plt.pie(readmitted_counts.values, labels=readmitted_counts.index, autopct='%1.1f%%')
plt.title('Class distribution of readmission')
plt.show()

In [ ]:
# Plot categorical feature cardinality

categorical_unique_counts = df[categorical_columns].nunique().sort_values(ascending=False)

top_cat = categorical_unique_counts.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_cat.index[::-1], top_cat.values[::-1])
plt.title('Top 10 categorical features by number of unique values')
plt.xlabel('Number of unique categories')
plt.ylabel('Feature')
plt.show()

In [ ]:
# Plot zero-heavy features

zero_counts = (df[numerical_columns] == 0).sum().sort_values(ascending=False)
top_zero = zero_counts.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_zero.index[::-1], top_zero.values[::-1])
plt.title('Top 10 numerical features with most zero values')
plt.xlabel('Number of zeros')
plt.ylabel('Feature')
plt.show()

## 2. Assemble and preprocess predictors

### Define the binary outcome and fixed data split

In [ ]:
# define the binary classification target

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Create a working copy for modelling
df_model = df.copy()

# Standardise the target values before mapping
readmitted_clean = df_model["readmitted"].astype(str).str.strip().str.upper()

# Define the binary target
# 0 = no readmission
# 1 = readmission within or after 30 days
y = readmitted_clean.map({
    "NO": 0,
    "<30": 1,
    ">30": 1
})

print("Original dataframe shape:", df_model.shape)
print("\nOriginal readmitted distribution:")
print(readmitted_clean.value_counts(dropna=False))

print("\nBinary target distribution:")
print(y.value_counts(dropna=False).sort_index())

print("\nBinary target proportion:")
print(y.value_counts(normalize=True, dropna=False).sort_index())

print("\nMissing labels after mapping:", y.isna().sum())

In [ ]:
# Define the feature matrix using all original predictors except the target
X = df_model.drop(columns=["readmitted"])

print("Feature matrix shape:", X.shape)
print("\nFirst 10 feature columns:")
print(X.columns[:10].tolist())

In [ ]:
from sklearn.model_selection import train_test_split

# Create a reduced dataset for computational efficiency
# 25% is used for training, 25% is used for testing, and the remaining 50% is discarded

X_temp, X_unused, y_temp, y_unused = train_test_split(
    X,
    y,
    test_size=0.5,
    stratify=y,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTraining class proportion:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTest class proportion:")
print(y_test.value_counts(normalize=True).sort_index())
# The historical split is encounter-level, not patient-disjoint.
shared_patients = set(X_train["patient_nbr"]) & set(X_test["patient_nbr"])
print("Patients shared between train and test:", len(shared_patients))
if SMOKE_TEST:
    X_train, _, y_train, _ = train_test_split(X_train, y_train, train_size=1200, stratify=y_train, random_state=42)
    X_test, _, y_test, _ = train_test_split(X_test, y_test, train_size=800, stratify=y_test, random_state=42)
    print("Smoke-only reduced sizes:", X_train.shape, X_test.shape)


In [ ]:
# Plot the binary target distribution
binary_counts = y.value_counts().sort_index()

plt.figure(figsize=(7, 5))
plt.bar(["No readmission (0)", "Readmission (1)"], binary_counts.values)
plt.xlabel("Binary target class")
plt.ylabel("Number of samples")
plt.title("Binary readmission class distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Compare class proportions across the full dataset, training set, and test set
proportion_df = pd.DataFrame({
    "Overall": y.value_counts(normalize=True).sort_index(),
    "Training": y_train.value_counts(normalize=True).sort_index(),
    "Test": y_test.value_counts(normalize=True).sort_index()
})

proportion_df.index = ["No readmission (0)", "Readmission (1)"]

ax = proportion_df.T.plot(kind="bar", figsize=(8, 5))
ax.set_xlabel("Dataset split")
ax.set_ylabel("Proportion")
ax.set_title("Class proportions across the full dataset, training set, and test set")
plt.xticks(rotation=0)
plt.legend(title="Class")
plt.tight_layout()
plt.show()

### Missing values and predictor removal

In [ ]:
# create working copies and standardise missing values

X_train_prep = X_train.copy()
X_test_prep = X_test.copy()

# Replace question-mark entries with standard missing values
X_train_prep = X_train_prep.replace("?", np.nan)
X_test_prep = X_test_prep.replace("?", np.nan)

print("Training set shape before feature removal:", X_train_prep.shape)
print("Test set shape before feature removal:", X_test_prep.shape)

In [ ]:
# Inspect missingness using the training set only

missing_ratio_train = X_train_prep.isna().mean().sort_values(ascending=False)

print("Top 15 features by missing ratio in the training set:")
print(missing_ratio_train.head(15))

In [ ]:
# Define features to remove

identifier_features = ["encounter_id", "patient_nbr"]
high_missing_features = missing_ratio_train[missing_ratio_train > 0.40].index.tolist()

features_to_drop = sorted(set(identifier_features + high_missing_features))

print("Identifier features to remove:")
print(identifier_features)

print("\nHigh-missing features to remove:")
print(high_missing_features)

print("\nAll features to remove in general preprocessing:")
print(features_to_drop)
# Predictor removal is chosen once on the outer training set, not refitted per CV fold.


In [ ]:
# Remove the selected features from both training and test sets

X_train_clean = X_train_prep.drop(columns=features_to_drop)
X_test_clean = X_test_prep.drop(columns=features_to_drop)

print("Training set shape after feature removal:", X_train_clean.shape)
print("Test set shape after feature removal:", X_test_clean.shape)

In [ ]:
# Separate diagnosis features for diagnosis grouping

diagnosis_features = [col for col in ["diag_1", "diag_2", "diag_3"] if col in X_train_clean.columns]

# Define feature lists for general preprocessing in general preprocessing
numerical_features = X_train_clean.drop(columns=diagnosis_features).select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train_clean.drop(columns=diagnosis_features).select_dtypes(include=["object"]).columns.tolist()

print("Diagnosis features reserved for diagnosis grouping:")
print(diagnosis_features)

print("\nNumber of numerical features for general preprocessing:", len(numerical_features))
print("Number of categorical features for general preprocessing:", len(categorical_features))

print("\nFirst 10 numerical features:")
print(numerical_features[:10])

print("\nFirst 10 categorical features:")
print(categorical_features[:10])

In [ ]:
# Compare missing ratio before and after feature removal

# Before cleaning (already computed)
missing_before = missing_ratio_train

# After cleaning
missing_after = X_train_clean.isna().mean().sort_values(ascending=False)

# Take top features before cleaning
top_features = missing_before.head(10).index

before_vals = missing_before.loc[top_features]
after_vals = missing_after.reindex(top_features).fillna(0)

x = np.arange(len(top_features))

plt.figure(figsize=(10, 6))
plt.barh(x - 0.2, before_vals.values, height=0.4, label="Before cleaning")
plt.barh(x + 0.2, after_vals.values, height=0.4, label="After cleaning")

plt.yticks(x, top_features)
plt.xlabel("Missing ratio")
plt.title("Missing ratio before and after feature removal (training set)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compare total number of missing values per feature

missing_count_before = X_train_prep.isna().sum().sort_values(ascending=False)
missing_count_after = X_train_clean.isna().sum().sort_values(ascending=False)

top_features = missing_count_before.head(10).index

before_vals = missing_count_before.loc[top_features]
after_vals = missing_count_after.reindex(top_features).fillna(0)

x = np.arange(len(top_features))

plt.figure(figsize=(10, 6))
plt.barh(x - 0.2, before_vals.values, height=0.4, label="Before cleaning")
plt.barh(x + 0.2, after_vals.values, height=0.4, label="After cleaning")

plt.yticks(x, top_features)
plt.xlabel("Number of missing values")
plt.title("Missing values before and after feature removal")
plt.legend()
plt.tight_layout()
plt.show()

### Group diagnosis codes into broader categories

In [ ]:
# inspect diagnosis feature complexity

for col in diagnosis_features:
    print(f"{col} - number of unique values:", X_train_clean[col].nunique())

In [ ]:
# Define a function to group diagnosis codes into broader clinical categories

def group_diagnosis(code):
    try:
        code = str(code)

        if code.startswith("V") or code.startswith("E"):
            return "other"

        code = float(code)

        if 390 <= code < 460:
            return "circulatory"
        elif 460 <= code < 520:
            return "respiratory"
        elif 520 <= code < 580:
            return "digestive"
        elif 250 <= code < 251:
            return "diabetes"
        elif 800 <= code < 1000:
            return "injury"
        elif 710 <= code < 740:
            return "musculoskeletal"
        elif 580 <= code < 630:
            return "genitourinary"
        elif 140 <= code < 240:
            return "neoplasms"
        else:
            return "other"

    except:
        return "unknown"

In [ ]:
# Apply diagnosis grouping to training and test sets

for col in diagnosis_features:
    X_train_clean[col + "_grouped"] = X_train_clean[col].apply(group_diagnosis)
    X_test_clean[col + "_grouped"] = X_test_clean[col].apply(group_diagnosis)

print("Grouped diagnosis features created:")
print([col + "_grouped" for col in diagnosis_features])

In [ ]:
# Remove original diagnosis features

X_train_final = X_train_clean.drop(columns=diagnosis_features)
X_test_final = X_test_clean.drop(columns=diagnosis_features)

print("Final training shape:", X_train_final.shape)
print("Final test shape:", X_test_final.shape)

In [ ]:
# Check number of categories after grouping

for col in [c + "_grouped" for c in diagnosis_features]:
    print(f"{col} - number of categories:", X_train_final[col].nunique())

In [ ]:
# Plot number of categories before and after grouping

before_counts = [
    X_train_clean[col].nunique() for col in diagnosis_features
]

after_counts = [
    X_train_final[col + "_grouped"].nunique() for col in diagnosis_features
]

labels = diagnosis_features

x = np.arange(len(labels))

plt.figure(figsize=(8, 5))
plt.bar(x - 0.2, before_counts, width=0.4, label="Before grouping")
plt.bar(x + 0.2, after_counts, width=0.4, label="After grouping")

plt.xticks(x, labels)
plt.ylabel("Number of categories")
plt.title("Reduction in diagnosis feature cardinality after grouping")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot distribution of grouped diagnosis features

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, col in enumerate(diagnosis_features):

    grouped_col = col + "_grouped"

    # Count category frequencies
    counts = X_train_final[grouped_col].value_counts().sort_values(ascending=True)

    axes[i].barh(counts.index, counts.values)
    axes[i].set_title(f"{col} disease group distribution")
    axes[i].set_xlabel("Number of samples")
    axes[i].set_ylabel("Disease group")

plt.tight_layout()
plt.show()

In [ ]:
# define the final feature lists and final preprocessor for modelling

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Define the final numerical and categorical feature lists
numerical_features_final = X_train_final.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features_final = X_train_final.select_dtypes(include=["object"]).columns.tolist()

print("Final numerical feature count:", len(numerical_features_final))
print("Final categorical feature count:", len(categorical_features_final))
print("\nGrouped diagnosis features included in final dataset:")
print([col for col in ["diag_1_grouped", "diag_2_grouped", "diag_3_grouped"] if col in X_train_final.columns])

# Define final preprocessing pipelines
numerical_transformer_final = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_final = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_final = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer_final, numerical_features_final),
        ("cat", categorical_transformer_final, categorical_features_final)
    ],
    remainder="drop"
)

print("\nFinal preprocessor has been defined.")
print("All features in X_train_final, including grouped diagnosis features, will be used for modelling.")

## 3. Compare classification pipelines

### Evaluation metrics

In [ ]:
# define evaluation metrics

from sklearn.metrics import (
    make_scorer,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

scoring = {
    "accuracy": make_scorer(accuracy_score),
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc"
}

primary_metric = "f1"

print("Metrics defined for model evaluation:")
for metric_name in scoring.keys():
    print("-", metric_name)

print("\nPrimary metric for model selection:", primary_metric)

### Candidate classifiers

In [ ]:
from sklearn.base import clone
# Shared score extraction must be defined before plotting or evaluation.
def get_model_scores(trained_model, X_data):
    if hasattr(trained_model, "decision_function"):
        return trained_model.decision_function(X_data)
    return trained_model.predict_proba(X_data)[:, 1]

# define the models for comparison

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "Linear SVM": SVC(kernel="linear", probability=False, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1)
}

baseline_model_name = "Linear SVM"

print("Models defined for comparison:")
for model_name in models.keys():
    print("-", model_name)

print("\nBaseline model:", baseline_model_name)

### Hyperparameter search with stratified cross-validation

In [ ]:
# define hyperparameter grids and cross-validation strategy

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

param_grids = {
    "Linear SVM": {
        "model__C": [0.1, 1]
    },
    "Logistic Regression": {
        "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
        "model__solver": ["liblinear"]
    },
    "Random Forest": {
        "model__n_estimators": [100],
        "model__max_depth": [2, 4, 6, 8, 10, 12],
        "model__min_samples_split": [2]
    }
}

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print("Parameter grids defined for all models.")
print("Cross-validation strategy:", cv_strategy)

for model_name, grid in param_grids.items():
    total_candidates = 1
    for values in grid.values():
        total_candidates *= len(values)
    total_fits = total_candidates * cv_strategy.get_n_splits()
    print(f"{model_name}: {total_candidates} parameter combinations, {total_fits} CV fits")
if SMOKE_TEST:
    param_grids = {
        "Linear SVM": {"model__C": [0.1]},
        "Logistic Regression": {"model__C": [1], "model__solver": ["liblinear"]},
        "Random Forest": {"model__n_estimators": [20], "model__max_depth": [6], "model__min_samples_split": [2]},
    }


In [ ]:
# build pipeline and grid search objects for each model

search_objects = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor_final)),
        ("model", clone(model))
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring=scoring,
        refit=primary_metric,
        cv=cv_strategy,
        n_jobs=1,
        verbose=4,
        return_train_score=True
    )

    search_objects[model_name] = grid_search

print("Grid search objects created for:")
for model_name in search_objects.keys():
    print("-", model_name)

In [ ]:
# run grid search with detailed progress tracking

import time

results_dict = {}

total_models = len(search_objects)

for i, (model_name, grid_search) in enumerate(search_objects.items(), 1):
    print("\n" + "=" * 70)
    print(f"Running model {i} of {total_models}: {model_name}")
    print("=" * 70)

    start_time = time.time()

    grid_search.fit(X_train_final, y_train)

    elapsed_time = time.time() - start_time

    results_dict[model_name] = {
        "search_object": grid_search,
        "best_estimator": grid_search.best_estimator_,
        "best_params": grid_search.best_params_,
        "best_cv_score": grid_search.best_score_,
        "cv_results": grid_search.cv_results_
    }

    print("\nFinished:", model_name)
    print("Best parameters:", grid_search.best_params_)
    print(f"Best mean CV {primary_metric}: {grid_search.best_score_:.4f}")
    print(f"Time taken: {elapsed_time / 60:.2f} minutes")

In [ ]:
# summarise the best cross-validation results

for model_name, result in results_dict.items():
    print(f"\nModel: {model_name}")
    print("Best parameters:", result["best_params"])
    print(f"Best mean CV {primary_metric}:", round(result["best_cv_score"], 4))

### Hyperparameter curves and held-out diagnostics

In [ ]:
# compare all models across five test-set metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd
import matplotlib.pyplot as plt

test_metric_rows = []

for model_name, result in results_dict.items():
    best_model = result["best_estimator"]
    y_pred = best_model.predict(X_test_final)
    y_score = get_model_scores(best_model, X_test_final)

    test_metric_rows.append({
        "Model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score)
    })

test_metrics_df = pd.DataFrame(test_metric_rows).set_index("Model")
display(test_metrics_df.round(4))

ax = test_metrics_df.plot(kind="bar", figsize=(10, 6))
ax.set_title("Test-set performance comparison across five metrics")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=0)
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

In [ ]:
# plot multi-metric hyperparameter performance and train-validation comparison

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc"]

param_column_map = {
    "Linear SVM": "param_model__C",
    "Logistic Regression": "param_model__C",
    "Random Forest": "param_model__max_depth"
}

fig, axes = plt.subplots(len(results_dict), 2, figsize=(14, 5 * len(results_dict)))

if len(results_dict) == 1:
    axes = np.array([axes])

for row_idx, (model_name, result) in enumerate(results_dict.items()):
    cv_results_df = pd.DataFrame(result["cv_results"]).copy()

    x_col = param_column_map.get(model_name)
    if x_col not in cv_results_df.columns:
        param_cols = [col for col in cv_results_df.columns if col.startswith("param_")]
        x_col = param_cols[0]

    x_numeric = pd.to_numeric(cv_results_df[x_col], errors="coerce")

    if x_numeric.notna().all():
        cv_results_df["_x_plot"] = x_numeric
        cv_results_df = cv_results_df.sort_values("_x_plot")
        x_values = cv_results_df["_x_plot"].values
    else:
        cv_results_df["_x_plot"] = cv_results_df[x_col].astype(str)
        x_values = cv_results_df["_x_plot"].values

    # Left panel: all validation metrics against the hyperparameter
    ax_left = axes[row_idx, 0]
    for metric_name in metric_names:
        ax_left.plot(
            x_values,
            cv_results_df[f"mean_test_{metric_name}"],
            marker="o",
            label=metric_name
        )
    ax_left.set_title(f"{model_name}: validation metrics vs hyperparameter")
    ax_left.set_xlabel(x_col.replace("param_model__", ""))
    ax_left.set_ylabel("Mean cross-validation score")
    ax_left.legend()

    # Right panel: train vs validation comparison for F1
    ax_right = axes[row_idx, 1]
    ax_right.plot(
        x_values,
        cv_results_df["mean_train_f1"],
        marker="o",
        label="Train F1"
    )
    ax_right.plot(
        x_values,
        cv_results_df["mean_test_f1"],
        marker="o",
        label="Validation F1"
    )
    ax_right.set_title(f"{model_name}: train vs validation F1")
    ax_right.set_xlabel(x_col.replace("param_model__", ""))
    ax_right.set_ylabel("Mean F1 score")
    ax_right.legend()

plt.tight_layout()
plt.show()

In [ ]:
# plot confusion matrices for all models

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(results_dict), figsize=(5 * len(results_dict), 4))

if len(results_dict) == 1:
    axes = [axes]

for ax, (model_name, result) in zip(axes, results_dict.items()):
    best_model = result["best_estimator"]
    y_pred = best_model.predict(X_test_final)

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        ax=ax,
        colorbar=False
    )
    ax.set_title(model_name)

plt.tight_layout()
plt.show()

In [ ]:
# plot Precision-Recall curves for all models

from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

average_precision_summary = {}

for model_name, result in results_dict.items():
    best_model = result["best_estimator"]
    y_score = get_model_scores(best_model, X_test_final)

    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_score)
    average_precision = average_precision_score(y_test, y_score)
    average_precision_summary[model_name] = average_precision

    plt.plot(
        recall_vals,
        precision_vals,
        label=f"{model_name} (AP = {average_precision:.3f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves for the selected models")
plt.legend()
plt.tight_layout()
plt.show()

print("Average precision values:")
for model_name, average_precision in average_precision_summary.items():
    print(f"{model_name}: {average_precision:.4f}")

### Cross-validation and held-out results

In [ ]:
# create a full summary table for cross-validation and test performance

import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

summary_rows = []

metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc"]

for model_name, result in results_dict.items():
    best_model = result["best_estimator"]
    cv_results_df = pd.DataFrame(result["cv_results"]).copy()

    # Select the best row directly from rank_test_f1
    best_row = cv_results_df.sort_values("rank_test_f1").iloc[0]

    # Compute test-set predictions and scores
    y_pred = best_model.predict(X_test_final)
    y_score = get_model_scores(best_model, X_test_final)

    row = {
        "Model": model_name,
        "Best parameters": str(result["best_params"])
    }

    # Add cross-validation mean ± std values
    for metric_name in metric_names:
        cv_mean = best_row[f"mean_test_{metric_name}"]
        cv_std = best_row[f"std_test_{metric_name}"]
        row[f"CV {metric_name.title()}"] = f"{cv_mean:.3f} ± {cv_std:.3f}"

    # Add single test-set values
    row["Test Accuracy"] = accuracy_score(y_test, y_pred)
    row["Test Precision"] = precision_score(y_test, y_pred, zero_division=0)
    row["Test Recall"] = recall_score(y_test, y_pred, zero_division=0)
    row["Test F1"] = f1_score(y_test, y_pred, zero_division=0)
    row["Test ROC-AUC"] = roc_auc_score(y_test, y_score)

    summary_rows.append(row)

results_summary_df = pd.DataFrame(summary_rows)

# Reorder columns for a cleaner final presentation
results_summary_df = results_summary_df[
    [
        "Model",
        "Best parameters",
        "CV Accuracy",
        "CV Precision",
        "CV Recall",
        "CV F1",
        "CV Roc_Auc",
        "Test Accuracy",
        "Test Precision",
        "Test Recall",
        "Test F1",
        "Test ROC-AUC"
    ]
]

# Round the test-set metrics
for col in ["Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC"]:
    results_summary_df[col] = results_summary_df[col].map(lambda x: round(x, 3))

print("Final performance summary table:")
display(results_summary_df)

## 4. Interpret model features

### Coefficients, tree importance, and permutation importance

In [ ]:
# model-specific feature importance (coefficients and feature_importances_)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

top_n = 10

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (model_name, result) in zip(axes, results_dict.items()):
    best_pipeline = result["best_estimator"]
    preprocessor = best_pipeline.named_steps["preprocessor"]
    model = best_pipeline.named_steps["model"]

    feature_names = preprocessor.get_feature_names_out()

    if model_name in ["Linear SVM", "Logistic Regression"]:
        importance_values = np.abs(model.coef_.ravel())
        x_label = "Absolute coefficient"
    elif model_name == "Random Forest":
        importance_values = model.feature_importances_
        x_label = "Feature importance"
    else:
        raise ValueError(f"Unsupported model: {model_name}")

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance_values
    }).sort_values("Importance", ascending=False).head(top_n)

    importance_df = importance_df.sort_values("Importance", ascending=True)

    ax.barh(importance_df["Feature"], importance_df["Importance"])
    ax.set_title(model_name)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Feature")

plt.tight_layout()
plt.show()

In [ ]:
# permutation importance for Random Forest (using training data)

from sklearn.inspection import permutation_importance
import pandas as pd
import matplotlib.pyplot as plt

top_n = 10

rf_pipeline = results_dict["Random Forest"]["best_estimator"]

perm_result = permutation_importance(
    rf_pipeline,
    X_train_final,
    y_train,
    scoring="f1",
    n_repeats=2 if SMOKE_TEST else 5,
    random_state=42,
    n_jobs=1
)

perm_importance_df = pd.DataFrame({
    "Feature": X_train_final.columns,
    "Importance": perm_result.importances_mean
}).sort_values("Importance", ascending=False).head(top_n)

perm_importance_df = perm_importance_df.sort_values("Importance", ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(perm_importance_df["Feature"], perm_importance_df["Importance"])
plt.xlabel("Mean permutation importance")
plt.ylabel("Feature")
plt.title("Random Forest permutation importance (top 10)")
plt.tight_layout()
plt.show()

print("Top features from Random Forest permutation importance:")
display(perm_importance_df.sort_values("Importance", ascending=False))

### Compare top features across models

In [ ]:
# top-feature overlap heatmap (wider version)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

top_n = 10

top_features_by_model = {}

for model_name, result in results_dict.items():
    best_pipeline = result["best_estimator"]
    preprocessor = best_pipeline.named_steps["preprocessor"]
    model = best_pipeline.named_steps["model"]

    feature_names = preprocessor.get_feature_names_out()

    if model_name in ["Linear SVM", "Logistic Regression"]:
        importance_values = np.abs(model.coef_.ravel())
    else:
        importance_values = model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance_values
    }).sort_values("Importance", ascending=False).head(top_n)

    top_features_by_model[model_name] = importance_df["Feature"].tolist()

all_top_features = sorted(set().union(*top_features_by_model.values()))

overlap_df = pd.DataFrame(0, index=all_top_features, columns=top_features_by_model.keys())

for model_name, feature_list in top_features_by_model.items():
    overlap_df.loc[feature_list, model_name] = 1

plt.figure(figsize=(8, max(4, len(all_top_features) * 0.22)))
plt.imshow(overlap_df.values, aspect="auto")

plt.xticks(range(len(overlap_df.columns)), overlap_df.columns, fontsize=9)
plt.yticks(range(len(overlap_df.index)), overlap_df.index, fontsize=8)

plt.title("Top feature overlap across models", fontsize=11)
plt.xlabel("Model", fontsize=10)
plt.ylabel("Feature", fontsize=10)

cbar = plt.colorbar(fraction=0.03, pad=0.02)
cbar.set_label("Included in top 10", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# pairwise overlap counts (compact full version)

import itertools
import pandas as pd
import matplotlib.pyplot as plt

pairwise_rows = []

for model_a, model_b in itertools.combinations(top_features_by_model.keys(), 2):
    overlap_count = len(set(top_features_by_model[model_a]) & set(top_features_by_model[model_b]))
    pairwise_rows.append({
        "Comparison": f"{model_a} vs {model_b}",
        "Shared top features": overlap_count
    })

all_three_overlap = len(
    set(top_features_by_model["Linear SVM"])
    & set(top_features_by_model["Logistic Regression"])
    & set(top_features_by_model["Random Forest"])
)

pairwise_rows.append({
    "Comparison": "All three models",
    "Shared top features": all_three_overlap
})

pairwise_df = pd.DataFrame(pairwise_rows)

plt.figure(figsize=(5, 3))
plt.bar(pairwise_df["Comparison"], pairwise_df["Shared top features"])

plt.title("Shared features", fontsize=10)
plt.xlabel("Model comparison", fontsize=9)
plt.ylabel("Count", fontsize=9)

plt.xticks(rotation=20, ha="right", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# shared vs unique features (compact full version)

import pandas as pd
import matplotlib.pyplot as plt

shared_unique_rows = []

all_feature_frequency = {}

for feature_list in top_features_by_model.values():
    for feature in feature_list:
        all_feature_frequency[feature] = all_feature_frequency.get(feature, 0) + 1

for model_name, feature_list in top_features_by_model.items():
    shared_count = sum(all_feature_frequency[feature] > 1 for feature in feature_list)
    unique_count = sum(all_feature_frequency[feature] == 1 for feature in feature_list)

    shared_unique_rows.append({
        "Model": model_name,
        "Shared": shared_count,
        "Unique": unique_count
    })

shared_unique_df = pd.DataFrame(shared_unique_rows).set_index("Model")

shared_unique_df.plot(
    kind="bar",
    stacked=True,
    figsize=(5, 3)
)

plt.title("Shared vs unique features", fontsize=10)
plt.xlabel("Model", fontsize=9)
plt.ylabel("Count", fontsize=9)

plt.xticks(rotation=0, fontsize=8)

plt.tight_layout()
plt.show()

### Summarise feature groups

In [ ]:
# feature category plots (compact full version)

import pandas as pd
import matplotlib.pyplot as plt

def categorise_feature(feature_name):
    feature_name = feature_name.lower()

    if "diag_" in feature_name:
        return "Diagnosis"
    elif "age" in feature_name or "race" in feature_name or "gender" in feature_name:
        return "Demographic"
    elif "admission" in feature_name or "discharge" in feature_name:
        return "Admission/Discharge"
    elif "insulin" in feature_name or "metformin" in feature_name or "glipizide" in feature_name \
         or "glyburide" in feature_name or "chlorpropamide" in feature_name \
         or "miglitol" in feature_name or "acarbose" in feature_name \
         or "pioglitazone" in feature_name or "rosiglitazone" in feature_name:
        return "Medication"
    elif "num_" in feature_name or "time_in_hospital" in feature_name:
        return "Healthcare utilisation"
    else:
        return "Other"

category_rows = []

for model_name, feature_list in top_features_by_model.items():
    for feature in feature_list:
        category_rows.append({
            "Model": model_name,
            "Category": categorise_feature(feature)
        })

category_df = pd.DataFrame(category_rows)

category_count_df = category_df.groupby(["Model", "Category"]).size().unstack(fill_value=0)

category_count_df.plot(
    kind="bar",
    stacked=True,
    figsize=(5, 3)
)

plt.title("Feature categories (count)", fontsize=10)
plt.xlabel("Model", fontsize=9)
plt.ylabel("Count", fontsize=9)

plt.xticks(rotation=0, fontsize=8)

plt.tight_layout()
plt.show()

category_prop_df = category_count_df.div(category_count_df.sum(axis=1), axis=0)

category_prop_df.plot(
    kind="bar",
    stacked=True,
    figsize=(5, 3)
)

plt.title("Feature categories (proportion)", fontsize=10)
plt.xlabel("Model", fontsize=9)
plt.ylabel("Proportion", fontsize=9)

plt.xticks(rotation=0, fontsize=8)

plt.tight_layout()
plt.show()

## 5. Evaluate random forest feature selection

### Select features inside the model pipeline

In [ ]:
# select the best model from baseline models

from sklearn.base import clone
import pandas as pd

selection_model_rows = []

for model_name, result in results_dict.items():
    selection_model_rows.append({
        "Model": model_name,
        "Best mean CV F1": result["best_cv_score"],
        "Best parameters": result["best_params"]
    })

selection_model_df = pd.DataFrame(selection_model_rows).sort_values(
    "Best mean CV F1",
    ascending=False
).reset_index(drop=True)

best_model_name_selection = "Random Forest"
best_model_params_selection = results_dict["Random Forest"]["best_params"]

print("baseline models model ranking by mean CV F1:")
display(selection_model_df)

print("Model family used for feature selection:", best_model_name_selection)
print("Best baseline models parameters:", best_model_params_selection)

best_model_selection = clone(models[best_model_name_selection])

best_model_param_dict = {
    param_name.replace("model__", ""): param_value
    for param_name, param_value in best_model_params_selection.items()
}

best_model_selection.set_params(**best_model_param_dict)

print("\nSelected model object for feature selection:")
print(best_model_selection)

In [ ]:
# define the alternative pipeline with SelectFromModel

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Use the best Random Forest settings from baseline models
best_rf_params = {
    key.replace("model__", ""): value
    for key, value in results_dict["Random Forest"]["best_params"].items()
}
best_rf_params.update(random_state=42, n_jobs=-1)

# Use the same Random Forest settings for both selector and final model
selector_model = RandomForestClassifier(**best_rf_params)
final_model = RandomForestClassifier(**best_rf_params)

alternative_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor_final)),
    ("selector", SelectFromModel(estimator=selector_model)),
    ("model", final_model)
])

selection_param_grid = {
    "selector__threshold": [0.001, 0.002, 0.005, "median", "mean"]
}

selection_cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

selection_search = GridSearchCV(
    estimator=alternative_pipeline,
    param_grid=selection_param_grid,
    scoring=scoring,
    refit=primary_metric,
    cv=selection_cv_strategy,
    n_jobs=1,
    verbose=4,
    return_train_score=True
)

print("Alternative pipeline with SelectFromModel has been defined.")
print("Feature-selection base model: Random Forest")
print("Feature-selection parameter grid:", selection_param_grid)
print("Cross-validation strategy:", selection_cv_strategy)
if SMOKE_TEST:
    selection_search.set_params(param_grid={"selector__threshold": ["mean"]})


In [ ]:
# run the alternative pipeline with SelectFromModel

import time
import pandas as pd

start_time = time.time()

selection_search.fit(X_train_final, y_train)

elapsed_time = time.time() - start_time

selection_results = {
    "selected_model_name": best_model_name_selection,
    "search_object": selection_search,
    "best_estimator": selection_search.best_estimator_,
    "best_params": selection_search.best_params_,
    "best_cv_score": selection_search.best_score_,
    "cv_results": selection_search.cv_results_
}

print("feature selection search completed.")
print("Selected base model:", best_model_name_selection)
print("Best feature-selection setting:", selection_search.best_params_)
print(f"Best mean CV {primary_metric}: {selection_search.best_score_:.4f}")
print(f"Time taken: {elapsed_time / 60:.2f} minutes")

selection_results_df = pd.DataFrame(selection_search.cv_results_)

display(
    selection_results_df[
        [
            "param_selector__threshold",
            "mean_test_f1",
            "std_test_f1",
            "mean_test_accuracy",
            "mean_test_roc_auc"
        ]
    ].sort_values("mean_test_f1", ascending=False)
)

### Bootstrap comparison of fitted models

In [ ]:
# create a comparison table for baseline models and feature selection
# with standard deviation for both cross-validation and test performance

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.utils import resample

def bootstrap_test_metrics(model, X_test_data, y_test_data, n_bootstrap=100, random_state=42):
    rng = np.random.RandomState(random_state)

    accuracy_list = []
    precision_list = []
    recall_list = []
    f1_list = []
    roc_auc_list = []

    X_test_array = X_test_data.copy()
    y_test_array = y_test_data.copy()

    for _ in range(n_bootstrap):
        sample_indices = rng.choice(len(y_test_array), size=len(y_test_array), replace=True)

        if hasattr(X_test_array, "iloc"):
            X_sample = X_test_array.iloc[sample_indices]
        else:
            X_sample = X_test_array[sample_indices]

        if hasattr(y_test_array, "iloc"):
            y_sample = y_test_array.iloc[sample_indices]
        else:
            y_sample = y_test_array[sample_indices]

        y_pred = model.predict(X_sample)
        y_score = get_model_scores(model, X_sample)

        accuracy_list.append(accuracy_score(y_sample, y_pred))
        precision_list.append(precision_score(y_sample, y_pred, zero_division=0))
        recall_list.append(recall_score(y_sample, y_pred, zero_division=0))
        f1_list.append(f1_score(y_sample, y_pred, zero_division=0))
        if len(np.unique(y_sample)) == 2:
            roc_auc_list.append(roc_auc_score(y_sample, y_score))

    return {
        "Test Accuracy": (np.mean(accuracy_list), np.std(accuracy_list)),
        "Test Precision": (np.mean(precision_list), np.std(precision_list)),
        "Test Recall": (np.mean(recall_list), np.std(recall_list)),
        "Test F1": (np.mean(f1_list), np.std(f1_list)),
        "Test ROC-AUC": (np.mean(roc_auc_list), np.std(roc_auc_list))
    }

# Retrieve the best Random Forest result from baseline models
baseline_result = results_dict["Random Forest"]
baseline_best_model = baseline_result["best_estimator"]
baseline_cv_results_df = pd.DataFrame(baseline_result["cv_results"]).copy()
baseline_best_row = baseline_cv_results_df.sort_values("rank_test_f1").iloc[0]

# Retrieve the best result from feature selection
selection_best_model = selection_results["best_estimator"]
selection_cv_results_df = pd.DataFrame(selection_results["cv_results"]).copy()
selection_best_row = selection_cv_results_df.sort_values("rank_test_f1").iloc[0]

comparison_rows = []

for model_label, best_model, best_row in [
    ("Random Forest (baseline models)", baseline_best_model, baseline_best_row),
    ("Random Forest + Feature Selection (feature selection)", selection_best_model, selection_best_row)
]:
    bootstrap_metrics = bootstrap_test_metrics(
        model=best_model,
        X_test_data=X_test_final,
        y_test_data=y_test,
        n_bootstrap=5 if SMOKE_TEST else 100,
        random_state=42
    )

    row = {
        "Model": model_label,
        "CV Accuracy": f"{best_row['mean_test_accuracy']:.3f} ± {best_row['std_test_accuracy']:.3f}",
        "CV Precision": f"{best_row['mean_test_precision']:.3f} ± {best_row['std_test_precision']:.3f}",
        "CV Recall": f"{best_row['mean_test_recall']:.3f} ± {best_row['std_test_recall']:.3f}",
        "CV F1": f"{best_row['mean_test_f1']:.3f} ± {best_row['std_test_f1']:.3f}",
        "CV ROC-AUC": f"{best_row['mean_test_roc_auc']:.3f} ± {best_row['std_test_roc_auc']:.3f}",
        "Test Accuracy": f"{bootstrap_metrics['Test Accuracy'][0]:.3f} ± {bootstrap_metrics['Test Accuracy'][1]:.3f}",
        "Test Precision": f"{bootstrap_metrics['Test Precision'][0]:.3f} ± {bootstrap_metrics['Test Precision'][1]:.3f}",
        "Test Recall": f"{bootstrap_metrics['Test Recall'][0]:.3f} ± {bootstrap_metrics['Test Recall'][1]:.3f}",
        "Test F1": f"{bootstrap_metrics['Test F1'][0]:.3f} ± {bootstrap_metrics['Test F1'][1]:.3f}",
        "Test ROC-AUC": f"{bootstrap_metrics['Test ROC-AUC'][0]:.3f} ± {bootstrap_metrics['Test ROC-AUC'][1]:.3f}"
    }

    comparison_rows.append(row)

selection_comparison_df = pd.DataFrame(comparison_rows)

print("Performance comparison between baseline models and feature selection:")
display(selection_comparison_df)

In [ ]:
# summarise the performance change from baseline models to feature selection
# using the mean test performance extracted from the formatted table

import pandas as pd

def extract_mean_from_mean_std(value):
    return float(str(value).split("±")[0].strip())

metric_change_df = pd.DataFrame({
    "Metric": ["Test Accuracy", "Test Precision", "Test Recall", "Test F1", "Test ROC-AUC"],
    "baseline models": [
        extract_mean_from_mean_std(selection_comparison_df.loc[0, "Test Accuracy"]),
        extract_mean_from_mean_std(selection_comparison_df.loc[0, "Test Precision"]),
        extract_mean_from_mean_std(selection_comparison_df.loc[0, "Test Recall"]),
        extract_mean_from_mean_std(selection_comparison_df.loc[0, "Test F1"]),
        extract_mean_from_mean_std(selection_comparison_df.loc[0, "Test ROC-AUC"])
    ],
    "feature selection": [
        extract_mean_from_mean_std(selection_comparison_df.loc[1, "Test Accuracy"]),
        extract_mean_from_mean_std(selection_comparison_df.loc[1, "Test Precision"]),
        extract_mean_from_mean_std(selection_comparison_df.loc[1, "Test Recall"]),
        extract_mean_from_mean_std(selection_comparison_df.loc[1, "Test F1"]),
        extract_mean_from_mean_std(selection_comparison_df.loc[1, "Test ROC-AUC"])
    ]
})

metric_change_df["Change (feature selection - baseline models)"] = (
    metric_change_df["feature selection"] - metric_change_df["baseline models"]
).round(3)

print("Change in mean test performance after adding feature selection:")
display(metric_change_df)

### Performance and dimensionality trade-off

In [ ]:
# plot comparison of test metrics (baseline models vs feature selection)

import matplotlib.pyplot as plt
import numpy as np

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]

baseline_values = [
    float(str(selection_comparison_df.loc[0, "Test Accuracy"]).split("±")[0].strip()),
    float(str(selection_comparison_df.loc[0, "Test Precision"]).split("±")[0].strip()),
    float(str(selection_comparison_df.loc[0, "Test Recall"]).split("±")[0].strip()),
    float(str(selection_comparison_df.loc[0, "Test F1"]).split("±")[0].strip()),
    float(str(selection_comparison_df.loc[0, "Test ROC-AUC"]).split("±")[0].strip())
]

selection_values = [
    float(str(selection_comparison_df.loc[1, "Test Accuracy"]).split("±")[0].strip()),
    float(str(selection_comparison_df.loc[1, "Test Precision"]).split("±")[0].strip()),
    float(str(selection_comparison_df.loc[1, "Test Recall"]).split("±")[0].strip()),
    float(str(selection_comparison_df.loc[1, "Test F1"]).split("±")[0].strip()),
    float(str(selection_comparison_df.loc[1, "Test ROC-AUC"]).split("±")[0].strip())
]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(6, 3))

plt.bar(x - width / 2, baseline_values, width, label="baseline models")
plt.bar(x + width / 2, selection_values, width, label="feature selection")

plt.xticks(x, metrics, rotation=30)
plt.ylabel("Score")
plt.title("Test performance comparison: baseline models vs feature selection")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# plot feature count before vs after selection

import matplotlib.pyplot as plt

# Number of features before selection (after preprocessing)
fitted_preprocessor = selection_results["best_estimator"].named_steps["preprocessor"]
X_transformed = fitted_preprocessor.transform(X_train_final)
n_features_before = X_transformed.shape[1]

# Number of features after selection
selector = selection_results["best_estimator"].named_steps["selector"]
n_features_after = selector.get_support().sum()

labels = ["Before selection", "After selection"]
values = [n_features_before, n_features_after]

plt.figure(figsize=(4, 3))
plt.bar(labels, values)

plt.ylabel("Number of features")
plt.title("Feature reduction after selection")

for i, v in enumerate(values):
    plt.text(i, v, str(v), ha="center", va="bottom")

plt.tight_layout()
plt.show()

print("Number of features before selection:", n_features_before)
print("Number of features after selection:", n_features_after)

## Interpretation and limitations

The result tables above are produced by the current run. Compare historical scores separately in `results/`; smoke-check scores are not comparable to the full experiment.

- The binary target combines both readmission intervals, and records represent encounters rather than unique patients.
- The original encounter-level split permits the same patient in training and testing. Patient-grouped or temporal validation is a separate follow-up experiment.
- Missingness-based column removal is selected once on the outer training set. Imputation, scaling, one-hot encoding, and feature selection are fitted within CV pipelines; the whole workflow is not fully fold-local.
- Numeric admission/discharge IDs retain their original numeric treatment. Diagnosis groups are a simplified mapping, and missing diagnosis values follow the preserved mapping into `other`.
- Permutation importance is computed on training data and is descriptive, not independent or causal evidence.
- Bootstrap standard deviations describe resampling of one held-out encounter set using fixed fitted models; they do not measure retraining or patient-level uncertainty.
- Clinical deployment, causal effects, fairness, and prospective validity have not been established.
